In [1]:
"""
This script was developed to parallel process preformatted time series of input data needed for
the Kljun et. al 2d flux footprint prediction code and ultimately create daily-ETo-weighted
footprint georeferenced footprint rasters. 

Checks are performed on the input data to handle data quality issues. The weighting method 
uses normalized hourly proportions of ASCE ETo computed from NLDAS v2 data for the closest cell.
NLDAS data is automatically downloaded using OpenDAP given Earthdata login info. Only days with 
5 or more hours of data (only from hours between 6:00AM to 8:00 PM) must exist in a day. 
Checks are performed to ensure final weighting procedure was successful at different steps of 
the process. 

This script is not intended to be used by others but to document a workflow that was employed for
scientific purposes.
"""
import nldas_via_giovanni as nldas
import calc_footprint_FFP_climatology as ffp
import footprint_funcs as ff
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import refet
from pyproj import CRS, Transformer
import xarray
import requests
import multiprocessing as mp
import math

__author__='John Volk'

In [2]:
# read metadata that has each sites' elevation used in ETr/ETo calcs
# AMF_meta_path = Path('master_flux_station_list.csv')
AMF_meta = pd.read_csv('stations_metadata.csv', index_col='SITE_ID')

In [3]:
# specify path with input CSV files for each station with 
# input time series of needed data, e.g. zm, u_star, L,...
in_dir = Path('input')
hourly_files = list(in_dir.glob('*.csv'))
print(hourly_files)

[PosixPath('input/US-Bi1.csv')]


In [4]:
def read_compiled_input(path):
    """
    Check if required input data exists in file and is formatted appropriately.
    
    Input files should be hourly or finer temporal frequency, drops hours
    without required input data. 
    """
    ret = None
    need_vars = {'latitude','longitude','ET_corr','wind_dir','u_star','sigma_v','zm','hc','d','L'}
    str_value_columns = ['IGBP_land_classification','secondary_veg_type']
    #don't parse dates first check if required inputs exist to save processing time
    df=pd.read_csv(path, index_col='date', parse_dates=False)
    cols = df.columns
    check_1 = need_vars.issubset(cols)
    check_2 = len({'u_mean','z0'}.intersection(cols)) >= 1 # need one or the other
    
    # if either test failed then insufficient input data for footprint, abort
    if not check_1 or not check_2:
        return (ret, None, None)
    
    ret = df
    ret.index = pd.to_datetime(df.index)

    # make it hourly data
    # ret = ret.resample('h').mean()  # this can be used only when numeric columns exists in database
    agg_methods = {}
    for col_name in ret.columns:
        if col_name in str_value_columns:
            agg_methods[col_name] = 'first'
        else: 
            agg_methods[col_name] = 'mean'
    ret = ret.resample('h').agg(agg_methods)

    lat,lon = ret[['latitude','longitude']].values[0]
    keep_vars = need_vars.union({'u_mean','z0','IGBP_land_classification','secondary_veg_type'})
    drop_vars = list(set(cols).difference(keep_vars))
    ret.drop(drop_vars, axis=1, inplace=True)
    ret.dropna(subset=['wind_dir','u_star','sigma_v','d','zm','L','ET_corr'], how='any', inplace=True)
    # print(ret.head())

    return ret, lat, lon

In [5]:
def runner(path):
    """
    Given path to time series of site hourly (or finer) input data,
    compute daily ETo weighted footprint rasters. 
    """
    station = path.stem
    elevation = AMF_meta.loc[station, 'ELEVATION_METERS']
    utc_offset = AMF_meta.loc[station, 'UTC_OFFSET']
    if pd.isna(elevation):
        print(f"ERROR: Elevation is not provided in the metadata. Skipping {station}!")
        return
    if pd.isna(utc_offset):
        print(f"ERROR: UTC Offset is not provided in the metadata. Skipping {station}!")
        return
        
    df, latitude, longitude = read_compiled_input(path)
    if df.empty: 
        print(f'Insufficient data exists. Skipping {station}!')
        return
    
    station_coord = (longitude, latitude)

    print(f"{station} coordinates: [{longitude}, {latitude}]")
    # print(df.head())
    # print(f"PROJ data directory: {proj.datadir.get_data_dir()}")
    
    # get EPSG code from lat,long, convert to UTM
    EPSG = 32700 - np.round((45+latitude)/90.0)*100+np.round((183+longitude)/6.0)
    EPSG = int(EPSG)
    in_proj = CRS('EPSG:4326')
    out_proj = CRS('EPSG:{}'.format(EPSG))
    transformer = Transformer.from_crs(in_proj, out_proj, always_xy=True)
    (station_x,station_y) = transformer.transform(*station_coord)
    print('original coordinates:',station_x,station_y)
    
    # move coord to snap centroid to 30m grid, minimal distortion
    rx = station_x % 15
    if rx > 7.5:
        station_x += (15-rx)
        # final coords should be odd factors of 15
        if (station_x / 15) % 2 == 0:
            station_x -= 15
    else:    
        station_x -= rx
        if (station_x / 15) % 2 == 0:
            station_x += 15
            
    ry = station_y % 15
    if ry > 7.5:
        print('ry > 7.5')
        station_y += (15-ry )
        if (station_y / 15) % 2 == 0:
            station_y -= 15
    else:
        print('ry <= 7.5')
        station_y -= ry
        if (station_y / 15) % 2 == 0:
            station_y += 15
    print('adjusted coordinates:',station_x,station_y)

    #Other model parameters
    #modify if needed
    h_s = 2000.    #Height of atmos. boundary layer [m] - assumed
    dx = 30.       #Model resolution [m]
    origin_d = 300. #Model bounds distance from origin [m]
    start_hr = 5    # hours from 1 to 24, inclusive
    end_hr = 17     # hours from 1 to 24, inclusive

    hours_array = np.arange(start_hr, end_hr+1)
    n_hrs = len(hours_array) 

    out_dir = Path('All_output')/'AMF'/f'{station}'

    if not out_dir.is_dir():
        out_dir.mkdir(parents=True, exist_ok=True)

    # get NLDAS data
    min_df_dt = df.index.min()
    max_df_dt = df.index.max()
    print(f"Input data ranges from {min_df_dt} to {max_df_dt}")
    start_dt_utc = min_df_dt - pd.Timedelta(hours=utc_offset)
    end_dt_utc = max_df_dt - pd.Timedelta(hours=utc_offset)
    # if NLDAS data file exists, use it.
    nldas_ts_inf = out_dir/ f'nldas_ETr.csv'
    get_data = True
    if nldas_ts_inf.is_file():
        nldas_df = pd.read_csv(nldas_ts_inf, index_col='date', parse_dates=True).sort_index()
        if start_dt_utc >= nldas_df.index.min() or end_dt_utc <= nldas_df.index.max():
            print(f"Using the existing NLDAS file, {nldas_ts_inf}")
            get_data = False
    elif get_data:
        # get data from NLDAS and calculate ET
        start_dt_utc_str = start_dt_utc.strftime("%Y-%m-%dT%H:%M:%S")
        end_dt_utc_str = end_dt_utc.strftime("%Y-%m-%dT%H:%M:%S")
        nldas_df = get_nldas2_ETo(station_coord, start_dt_utc_str, end_dt_utc_str, elevation, out_dir)
        nldas_df = nldas_df.sort_index()
    # change UTC timestamp to station local timestamp
    nldas_df.index = nldas_df.index + pd.Timedelta(hours=utc_offset)
    # use only set hours
    nldas_df = nldas_df.between_time(f'{start_hr:02}:00', f'{end_hr:02}:00')

    #Loop through each day in the dataframe
    for date, day_df in df.groupby(df.index.date):
        #Subset dataframe to only values in day of year
        print(f'Date: {date}')
        day_df = day_df.between_time(f'{start_hr:02}:00', f'{end_hr:02}:00')  # <- inclusive
        
        # check on n hours per day`
        if len(day_df) < 5:
            print(f'Less than 5 hours of data on {date}, skipping.')
            continue
            
        new_dat = None

        out_f = out_dir/ f'{date}.tif'

        final_outf = out_dir/f'{date.year}-{date.month:02}-{date.day:02}_weighted.tif'
        if final_outf.is_file():
            print(f'final daily weighted footprint already wrote to: {final_outf}\nskipping.')
            continue # do not overwrite date/site raster 

        # make hourly band raster for the day
        missing_hours = []
        for indx, hour in enumerate(hours_array):

            band = indx + 1
            print(f'Hour: {hour}')

            try:
                temp_line = day_df.loc[day_df.index.hour == hour,:]
                if temp_line.empty: 
                    missing_hours.append(hour)
                    print(f'Missing all data for {date,hour} skipping')
                    continue
                zm = temp_line.zm.values - temp_line.d.values
                z0 = temp_line.z0.values if 'z0' in temp_line.columns else None
                u_mean = temp_line.u_mean.values if 'u_mean' in temp_line.columns else None
                if u_mean is not None: z0 = None
                # print(f'zm: {zm}, z0: {z0}, u_mean: {u_mean}')

                #Calculate footprint
                temp_ffp = ffp.ffp_climatology(domain=[-origin_d,origin_d,-origin_d,origin_d],dx=dx,dy=dx,
                                        zm=zm, h=h_s, rs=None, z0=z0, 
                                        ol=temp_line['L'].values,sigmav=temp_line['sigma_v'].values,
                                        ustar=temp_line['u_star'].values, umean=u_mean,
                                        wind_dir=temp_line['wind_dir'].values,
                                        crop=0,fig=0,verbosity=0)
                f_2d = np.array(temp_ffp['fclim_2d'])
                x_2d = np.array(temp_ffp['x_2d']) + station_x
                y_2d = np.array(temp_ffp['y_2d']) + station_y
                f_2d = f_2d*dx**2
                # print(f'f_2d: {f_2d}, x_2d: {x_2d}, y_2d: {y_2d}')

                #Calculate affine transform for given x_2d and y_2d
                affine_transform = ff.find_transform(x_2d,y_2d)

                #Create data file if not already created
                if new_dat is None:
                    #print(f_2d.shape)
                    new_dat = rasterio.open(
                        out_f,'w',driver='GTiff',dtype=rasterio.float64,
                        count=n_hrs,height=f_2d.shape[0],width=f_2d.shape[1],
                        transform=affine_transform, crs=out_proj.srs,
                        nodata=0.00000000e+000
                    )

            except Exception as e:
                print(f'Hour {hour} footprint failed, band {band} not written.')
                print(f'Exception: {e}')

                temp_ffp = None

                continue

            #Mask out points that are below a % threshold (defaults to 90%)
            f_2d = ff.mask_fp_cutoff(f_2d)

            #Write the new band
            new_dat.write(f_2d, band)

            #Update tags with metadata
            tag_dict = {'hour':f'{hour*100:04}',
                        'wind_dir':temp_line['wind_dir'].values,
                        'total_footprint':np.nansum(f_2d)}

            new_dat.update_tags(band,**tag_dict)

        #Close dataset if it exists
        try:
            new_dat.close()
        except:
            print(f'ERROR: could not write footprint for site: {station}:\nto: {out_f}')
            continue # skip to next day...

        # do hourly weighting - do not necessarily need to do this all in the same loop
        src = rasterio.open(out_f)
        # print(src)
        # hourly fetch scalar sums
        global_sum = np.zeros(shape=(n_hrs))
        # reading band in raster and normalized fetch rasters
        normed_fetch_rasters = [] 
        for band in range(1,n_hrs+1):  
            arr = src.read(band)
            global_sum[band-1] = arr.sum()
            if global_sum[band-1] == 0:
                tmp = np.zeros_like(arr)
            else:        
                tmp = arr / global_sum[band-1]
            normed_fetch_rasters.append(tmp)

        ETo = nldas_df.loc[nldas_df.index.date == date, 'ETo']
        min_max_normed_ETo = (ETo-min(ETo))/(max(ETo)-min(ETo)) # deal with negative ETo value proportions
        # take out hours where footprint does not exist
        i = 0
        for e, s in zip(min_max_normed_ETo.values, global_sum):
            if s == 0:
                min_max_normed_ETo.iloc[i] = 0
            i+=1
        # after removing hours now calculate hourly proportions
        nldas_df.loc[nldas_df.index.date == date, 'ETo_hr_props'] = min_max_normed_ETo / min_max_normed_ETo.sum()
        # weight normed hourly fetch rasters by hourly ETo proportions
        for i,hour in enumerate(hours_array): # everything here is hours 0-23
            if hour in missing_hours:
                normed_fetch_rasters[i] = normed_fetch_rasters[i] * 0
            else:
                normed_fetch_rasters[i] =\
                    normed_fetch_rasters[i] * nldas_df.loc[
                        (nldas_df.index.date == date) & (nldas_df.index.hour == hour), 'ETo_hr_props'
                ].values[0]
        # save hourly proportions to time series file
        # nldas_df.round(4).to_csv(nldas_ts_outf)

        # Last calculation, sum the weighted hourly rasters to a single daily fetch raster
        final_footprint = sum(normed_fetch_rasters)
        if not np.isclose(final_footprint.sum(), 1):
            print(f'check 1 failed! {final_footprint.sum()}\n')
            return
        # assert np.isclose(final_footprint.sum(), 1), f'check 1 failed! {final_footprint.sum()}\n{temp_line}'
        # next check
        for index, raster in enumerate(normed_fetch_rasters):
            hour = index + start_hr
            if hour not in missing_hours:
                if not np.isclose(
                    nldas_df.loc[
                        (nldas_df.index.date == date) & (nldas_df.index.hour == hour), 'ETo_hr_props'
                    ].values[0], raster.sum()
                ):
                    print(f'check 2 failed for hour {hour}!!!')
                    return
            """
            assert np.isclose(
                nldas_df.loc[
                    (nldas_df.index.date == date) & (nldas_df.index.hour == hour), 'ETo_hr_props'
                ].values[0], raster.sum()
            ), f'check 2 failed for hour {hour}!!!'
            """

        # finally, write daily corrected raster with UTM zone reference 
        corr_raster_path = final_outf
        out_raster = rasterio.open(
            corr_raster_path,'w',driver='GTiff',dtype=rasterio.float64,
            count=1,height=final_footprint.shape[0],width=final_footprint.shape[1],
            transform=src.transform, crs=out_proj.srs, nodata=0.00000000e+000
        )
        out_raster.write(final_footprint,1)
        out_raster.close()

    if 'ETo_hr_props' in nldas_df:
        nldas_ts_outf = out_dir/ f'nldas_ETr_props.csv'
        nldas_df.round(4).to_csv(nldas_ts_outf)

    print(f"{station} is done!")


In [6]:
def get_nldas2_ETo(
    coords: tuple[float, float], 
    start_dt_utc: str,  # Date must be the format of YYYY-MM-DDThh:mm:ss in UTC
    end_dt_utc: str,
    elevation: float,
    out_dir: str,
    api_token=""      # api_token can be empty if you have three files, .netrc, .dodsrc and .urs_cookies, created in the root direcotry
):
    zm = 10  # nldas2 windspeed height is 10 m  <- Used in refET calculation 
    # Date must be the format of YYYY-MM-DDThh:mm:ss in UTC
    # start_date = start_date.strftime("%Y-%m-%dT%H:%M:%S")
    # end_date = end_date.strftime("%Y-%m-%dT%H:%M:%S")
    print(f"Data is requested to NLDAS from {start_dt_utc} to {end_dt_utc}")

    # api request information    
    data = "NLDAS_FORA0125_H_2_0"
    variables = ["Rainf", "LWdown", "SWdown", "PotEvap", "PSurf", "Qair", "Tair", "Wind_E", "Wind_N"]
    """
    # variable infomation for NLDAS2
    variable_names = ["Precipitation hourly total", "Surface DW longwave radiation flux", "Surface DW shortwave radiation flux", 
                      "Potential evaporation", "Surface pressure", "2-m above ground specific humidity", "2-m above ground temperature",
                      "10-m above ground zonal wind", "10-m above ground meridional wind"]
    variable_units = ["kg/m2", "W/m2", "W/m2", "kg/m2", "Pa", "kg/kg", "K", "m/s", "m/s"]
    """

    nldas_df = nldas.get_nldas2_data(coords[1], coords[0], start_dt_utc, end_dt_utc, data, variables, out_dir, api_token)
    nldas_df.index.name = 'date'
    # print(nldas_df.head())

    
    nldas_df['pair'] = nldas_df['PSurf'] / 1000 # nldas air pres in Pa convert to kPa
    # sph = ds.get('SPF_H_110_HTGL').data # kg/kg
    nldas_df['ea'] = refet.calcs._actual_vapor_pressure(q=nldas_df['Qair'].to_numpy(), pair=nldas_df['pair'].to_numpy())  # ea in kPa
    # calculate hourly wind
    nldas_df['wind'] = np.sqrt(nldas_df['Wind_E'] ** 2 + nldas_df['Wind_N'] ** 2)
    # get temp convert to C
    nldas_df['temp'] = nldas_df['Tair'] - 273.15
    # get rs
    unit_dict = {'rs': 'w/m2'}

    nldas_df['doy'] = nldas_df.index.dayofyear
    nldas_df['HH'] = nldas_df.index.hour
    
    # create refet object for calculating
    for index, row in enumerate(nldas_df.itertuples()):

        etr_calculator = refet.Hourly(
            tmean=row.temp,
            ea=row.ea,
            rs=row.SWdown,
            uz=row.wind,
            zw=zm,
            elev=elevation,
            lat=coords[1],
            lon=coords[0],
            doy=row.doy,
            time=row.HH,
            method='asce',
            input_units=unit_dict
        )  # HH must be int
        # Calculate the ETr and store it in the DataFrame
        nldas_df.loc[row.Index, 'ETr'] = etr_calculator.etr()[0]
        nldas_df.loc[row.Index, 'ETo'] = etr_calculator.eto()[0]
    
    # Save     
    ETr_df = pd.DataFrame(columns=['ETr','ETo','ea','sph','wind','pair','temp','rs'])
    ETr_df['ETr'] = nldas_df['ETr']
    ETr_df['ETo'] = nldas_df['ETo']
    ETr_df['ea'] = nldas_df['ea']
    ETr_df['sph'] = nldas_df['Qair']
    ETr_df['wind'] = nldas_df['wind']
    ETr_df['pair'] = nldas_df['pair']
    ETr_df['temp'] = nldas_df['temp']
    ETr_df['rs'] = nldas_df['SWdown']
    ETr_df.index.name = 'date'
    # print(ETr_df.head())
    
    # save data
    nldas_ts_outf = out_dir/ f'nldas_ETr.csv'
    ETr_df.round(4).to_csv(nldas_ts_outf)
    
    return ETr_df
    


In [7]:
#pool = mp.Pool(processes=8)
#pool.map(runner,hourly_files)
for hourly_file in hourly_files:
    runner(hourly_file)
print("ALL DONE!")

US-Bi1 coordinates: [-121.49933, 38.0991538]
original coordinates: 631580.8467393691 4217879.800941323
ry > 7.5
adjusted coordinates: 631575.0 4217865.0
Input data ranges from 2016-08-13 00:00:00 to 2016-10-31 23:00:00
Data is requested to NLDAS from 2016-08-13T08:00:00 to 2016-11-01T07:00:00
Requesting Rainf data
Request for Rainf data is done.
Requesting LWdown data
Request for LWdown data is done.
Requesting SWdown data
Request for SWdown data is done.
Requesting PotEvap data
Request for PotEvap data is done.
Requesting PSurf data
Request for PSurf data is done.
Requesting Qair data
Request for Qair data is done.
Requesting Tair data
Request for Tair data is done.
Requesting Wind_E data
Request for Wind_E data is done.
Requesting Wind_N data
Request for Wind_N data is done.
NLDAS data is saved in All_output/AMF/US-Bi1/nldas.csv
Date: 2016-08-13
Hour: 5
Hour: 6
Hour: 7
Hour: 8
Hour: 9
Hour: 10
Hour: 11
Hour: 12
Hour: 13
Hour: 14
Hour: 15
Hour: 16
Hour: 17
Date: 2016-08-14
Hour: 5
Hou